In [ ]:
from sklearn.model_selection import train_test_split
import os
import pandas as pd
from sklearn.manifold import TSNE

In [ ]:
d = {
    "even": {
        0: 1,
        1: 0,
        2: 1,
        3: 0,
        4: 1,
        5: 0,
        6: 1,
        7: 0,
        8: 1,
        9: 0
    },
    "greater_4": {
        0: 0,
        1: 0,
        2: 0,
        3: 0,
        4: 0,
        5: 1, 
        6: 1,
        7: 1,
        8: 1,
        9: 1
    },
    "multiple_3": {
        0: 1,
        1: 0,
        2: 0,
        3: 1, 
        4: 0,
        5: 0,
        6: 1,
        7: 0,
        8: 0,
        9: 1
    }, 
    "prime": {
        0: 0,
        1: 1,
        2: 1,
        3: 1,
        4: 0,
        5: 1,
        6: 0,
        7: 1,
        8: 0,
        9: 0
    },
    "not_outlier": {
        0: 0,
        1: 0,
        2: 1,
        3: 1,
        4: 1,
        5: 1, 
        6: 1,
        7: 0,
        8: 0,
        9: 0
    }
}

In [ ]:
count = {i: 0 for i in range(10)}
for k in d:
    for i in range(10):
        if d[k][i] == 1:
            count[i] += 1

In [ ]:
count

In [ ]:
from sklearn.datasets import fetch_openml

# Scarica il dataset MNIST
mnist = fetch_openml('mnist_784', version=1)

In [ ]:
df = mnist["data"]

In [ ]:
df["target"] = mnist["target"]

In [ ]:
df["target"] = df["target"].astype(int)

In [ ]:
df["task1"] = df["target"].apply(lambda x : d["even"][x])
df["task2"] = df["target"].apply(lambda x : d["greater_4"][x])
df["task3"] = df["target"].apply(lambda x : d["multiple_3"][x])
df["task4"] = df["target"].apply(lambda x : d["prime"][x])
df["task5"] = df["target"].apply(lambda x : d["not_outlier"][x])

In [ ]:
for i in range(1, 785):
    df[f"pixel{i}"] = df[f"pixel{i}"] / 255

In [ ]:
df = df.sample(frac=1).reset_index(drop=True)

In [ ]:
train_df, test_df = train_test_split(df, test_size=0.1667, stratify=df['target'])
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

In [ ]:
root = "/Users/federicogiannini/Library/CloudStorage/OneDrive-PolitecnicodiMilano/SML_CL"

In [ ]:
df.to_csv(os.path.join(root, "datasets", "mnist.csv"), index=False)
train_df.to_csv(os.path.join(root, "datasets", "mnist_train.csv"), index=False)
test_df.to_csv(os.path.join(root, "datasets", "mnist_test.csv"), index=False)

In [1]:
from sklearn.model_selection import train_test_split
import os
import pandas as pd
import umap

2024-10-25 17:45:30.793504: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
root = "/Users/federicogiannini/Library/CloudStorage/OneDrive-PolitecnicodiMilano/SML_CL"

df = pd.read_csv(os.path.join(root, "datasets", "mnist.csv"))
train_df = pd.read_csv(os.path.join(root, "datasets", "mnist_train.csv"))
test_df = pd.read_csv(os.path.join(root, "datasets", "mnist_test.csv"))

In [3]:
umap_model = umap.UMAP(n_components=30)
umap_model.fit(train_df.iloc[:,:784].values)

UMAP(n_components=30, tqdm_kwds={'bar_format': '{desc}: {percentage:3.0f}%| {bar} {n_fmt}/{total_fmt} [{elapsed}]', 'desc': 'Epochs completed', 'disable': True})

In [4]:
X_train_red = umap_model.transform(train_df.iloc[:,:784].values)
X_test_red = umap_model.transform(test_df.iloc[:,:784].values)

In [5]:
def red(X, df):
    df_red = pd.DataFrame(X, columns=[f"feat{i}" for i in range(X.shape[1])])
    df_red["target"] = list(df["target"])
    for i in range(1,6):
        df_red[f"task{i}"] = list(df[f"task{i}"])
    return df_red

In [6]:
df_train_red = red(X_train_red, train_df)
df_test_red = red(X_test_red, test_df)
df_train_red.to_csv(os.path.join(root, "datasets", "mnist_red30_train.csv"), index=False)
df_test_red.to_csv(os.path.join(root, "datasets", "mnist_red30_test.csv"), index=False)

In [7]:
from dataset_utils import *
from sklearn.model_selection import train_test_split
import os
import pandas as pd
from sklearn.manifold import TSNE
import umap

In [9]:
df_train_red = pd.read_csv(os.path.join(root, "datasets", "mnist_red30_train.csv"))
df_test_red = pd.read_csv(os.path.join(root, "datasets", "mnist_red30_test.csv"))

In [10]:
set1 = [0,2,4,6,8]
set2 = [1,3,5,7,9]

In [11]:
dataset = "mnist_red30"

In [12]:
exps_train = make_exp_incremental_train(root, dataset, df_train_red, set1, set2)
exps_test = make_exp_incremental_test(root, dataset, df_test_red)

[(0, 1), (8, 9), (2, 7), (4, 3), (6, 5)] ACCEPTED
[(0, 5), (8, 7), (4, 1), (2, 3), (6, 9)] ACCEPTED
[(6, 5), (0, 3), (8, 9), (2, 7), (4, 1)] ACCEPTED
[(0, 7), (6, 3), (4, 9), (2, 1), (8, 5)] ACCEPTED
[(0, 7), (8, 1), (4, 3), (2, 9), (6, 5)] ACCEPTED
[(6, 9), (2, 1), (4, 5), (0, 7), (8, 3)] ACCEPTED
[(4, 9), (6, 1), (2, 7), (8, 5), (0, 3)] ACCEPTED
[(0, 1), (2, 3), (4, 5), (6, 9), (8, 7)] ACCEPTED
[(0, 1), (6, 9), (8, 3), (2, 5), (4, 7)] ACCEPTED
[(0, 9), (4, 5), (8, 1), (6, 7), (2, 3)] ACCEPTED


In [13]:
exps_sml_train = make_exp_sml_train(root, dataset, df_train_red)
exps_sml_test = make_exp_sml_test(root, dataset, df_test_red)

(1, 3, 4, 2, 5) ACCEPTED
(1, 5, 3, 4, 2) ACCEPTED
(1, 5, 3, 2, 4) ACCEPTED
(3, 2, 5, 1, 4) ACCEPTED
(5, 4, 2, 1, 3) ACCEPTED
(2, 5, 4, 1, 3) ACCEPTED
(2, 4, 3, 5, 1) ACCEPTED
(5, 4, 3, 1, 2) ACCEPTED
(5, 2, 4, 1, 3) ACCEPTED
(1, 5, 2, 3, 4) ACCEPTED


In [26]:
check_distr(exps_train)
check_distr(exps_test)
check_distr(exps_sml_train)
check_distr(exps_sml_test)